In [ ]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
import plotly.io as pio
from umap import UMAP
from hdbscan import HDBSCAN
from pathlib import Path
import time
import stopwordsiso as stopwords
from src.evaluation import TopicEvaluator

In [65]:
DATA_PATH = (
    r"D:\proposal\thiese\code\.venv\data\raw\isna_complete.csv"
)

In [3]:
df = pd.read_csv(DATA_PATH)

documents = (
    df["content"]
    .dropna()
    .astype(str)
    .tolist()
)

print(f"Number of documents: {len(documents)}")

Number of documents: 5969


In [64]:
df.columns

Index(['category', 'sub_category', 'title', 'url', 'content', 'published date',
       'summary', 'news_code'],
      dtype='object')

In [4]:
print("Dataset Shape:")
print(df.shape)

print()

print("Missing Values:")
print(df.isnull().sum())

print()

print("Categories:")
print(df["category"].value_counts())

Dataset Shape:
(5969, 8)

Missing Values:
category          0
sub_category      0
title             0
url               0
content           0
published date    0
summary           0
news_code         0
dtype: int64

Categories:
category
ورزشی            1396
سیاسی            1297
اقتصادی          1266
فرهنگی و هنری    1195
اجتماعی           815
Name: count, dtype: int64


In [6]:
embedding_model = SentenceTransformer(
    "HooshvareLab/bert-base-parsbert-uncased"
)

No sentence-transformers model found with name HooshvareLab/bert-base-parsbert-uncased. Creating a new one with mean pooling.


In [7]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

In [8]:
cluster_model = HDBSCAN(
    min_cluster_size=20,
    metric="euclidean",
    prediction_data=True
)

In [25]:
persian_stopwords = list(
    stopwords.stopwords("fa")
)

persian_stopwords.extend([
    "ایسنا",
    "گزارش"
])

vectorizer_model = CountVectorizer(
    stop_words=persian_stopwords,
    ngram_range=(1, 2),
    min_df=5
)

In [26]:
topic_model = BERTopic(

    embedding_model=embedding_model,

    umap_model=umap_model,

    hdbscan_model=cluster_model,

    vectorizer_model=vectorizer_model,

    language=None,

    calculate_probabilities=True,

    verbose=True
)

In [27]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

print(output_dir)

outputs


In [28]:
start = time.time()

topics, probabilities = topic_model.fit_transform(
    documents
)

end = time.time()

print(f"Execution Time: {end - start:.2f} seconds")

2026-07-10 19:34:18,843 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 187/187 [02:39<00:00,  1.17it/s]
2026-07-10 19:36:59,168 - BERTopic - Embedding - Completed ✓
2026-07-10 19:36:59,169 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-10 19:37:29,355 - BERTopic - Dimensionality - Completed ✓
2026-07-10 19:37:29,356 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-10 19:37:32,582 - BERTopic - Cluster - Completed ✓
2026-07-10 19:37:32,598 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-10 19:37:44,083 - BERTopic - Representation - Completed ✓


Execution Time: 208.84 seconds


In [ ]:
topic_info = topic_model.get_topic_info()

document_info = topic_model.get_document_info(
    documents
)
topic_freq = topic_model.get_topic_freq()

In [ ]:
display(topic_info.head())

display(document_info.head())

display(topic_freq.head())

In [ ]:
topic_info.to_csv(
    "../results/tables/topic_info.csv",
    index=False,
    encoding="utf-8-sig"
)

document_info.to_csv(
    "../results/tables/document_info.csv",
    index=False,
    encoding="utf-8-sig"
)

topic_freq.to_csv(
    "../results/tables/topic_frequency.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
print(topic_info.columns)

print(document_info.columns)

print(topic_freq.columns)

In [31]:
topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1537,-1_ایران_کشور_ملی_تیم,"[ایران, کشور, ملی, تیم, پیام, انتهای, انتهای پ...",[به گزارش ایسنا، حجت الاسلام موحدی آزاد دادستا...
1,0,1015,0_فیلم_کتاب_آثار_موسیقی,"[فیلم, کتاب, آثار, موسیقی, اثر, جشنواره, نمایش...",[به دلیل همزمانی با رحلت امام خمینی(ره)، نمایش...
2,1,467,1_نفت_شرکت_افزایش_دلار,"[نفت, شرکت, افزایش, دلار, هوش, مصنوعی, هوش مصن...",[به گزارش ایسنا، بهای معاملات وست تگزاس اینترم...
3,2,270,2_دولت_مجلس_کشور_شورای,"[دولت, مجلس, کشور, شورای, دارو, بودجه, اقتصادی...",[به گزارش ایسنا، مجتبی یوسفی در نشست خبری در ت...
4,3,252,3_قضایی_قوه_قضاییه_قوه قضاییه,"[قضایی, قوه, قضاییه, قوه قضاییه, رسیدگی, قانون...",[به گزارش ایسنا، حجت الاسلام و المسلمین خلیلی ...
5,4,187,4_اسلامی_انقلاب_ملت_ایران,"[اسلامی, انقلاب, ملت, ایران, حضرت, شهید, انقلا...",[به گزارش ایسنا، متن پیام ذبیحالله خدائیان، رئ...
6,5,180,5_برق_مصرف_آب_انرژی,"[برق, مصرف, آب, انرژی, تولید, کشور, صنعت, نیرو...",[به گزارش ایسنا، مصطفی رجبی مشهدی- معاون برق و...
7,6,145,6_بازی_تیم_بازیکنان_فوتبال,"[بازی, تیم, بازیکنان, فوتبال, استقلال, باشگاه,...",[به گزارش ایسنا و به نقل از رسانه رسمی باشگاه ...
8,7,124,7_آموزش_آموزشی_پرورش_آموزش پرورش,"[آموزش, آموزشی, پرورش, آموزش پرورش, مدارس, دان...",[به گزارش ایسنا، علیرضا کاظمی در برنامه گفتوگو...
9,8,114,8_سپاه_ارتش_عملیات_فروند,"[سپاه, ارتش, عملیات, فروند, سپاه پاسداران, آمر...",[به گزارش ایسنا، سرهنگ دوم پاسدار ابراهیم ذوال...


In [46]:
document_info.head()

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,به گزارش ایسنا، با اعلام فدراسیون فوتبال، تیم ...,-1,-1_ایران_کشور_ملی_تیم,"[ایران, کشور, ملی, تیم, پیام, انتهای, انتهای پ...",[به گزارش ایسنا، حجت الاسلام موحدی آزاد دادستا...,ایران - کشور - ملی - تیم - پیام - انتهای - انت...,0.422309,False
1,به گزارش ایسنا، اکبر کارگرجم بازیکن سابق استقل...,12,12_باشگاه_استقلال_پرسپولیس_ورزشگاه,"[باشگاه, استقلال, پرسپولیس, ورزشگاه, تیم, فوتب...",[به گزارش ایسنا، تیم فوتبال استقلال روز بیستوی...,باشگاه - استقلال - پرسپولیس - ورزشگاه - تیم - ...,0.385935,False
2,به گزارش ایسنا، هفته پنجم رقابتهای لیگ آزادگان...,-1,-1_ایران_کشور_ملی_تیم,"[ایران, کشور, ملی, تیم, پیام, انتهای, انتهای پ...",[به گزارش ایسنا، حجت الاسلام موحدی آزاد دادستا...,ایران - کشور - ملی - تیم - پیام - انتهای - انت...,0.852754,False
3,طبق اعلام نقی نصیری، مدیرعامل باشگاه مس سونگون...,12,12_باشگاه_استقلال_پرسپولیس_ورزشگاه,"[باشگاه, استقلال, پرسپولیس, ورزشگاه, تیم, فوتب...",[به گزارش ایسنا، تیم فوتبال استقلال روز بیستوی...,باشگاه - استقلال - پرسپولیس - ورزشگاه - تیم - ...,0.374967,False
4,این مسابقه از هفته پنجم لیگ برتر فوتبال کشور، ...,-1,-1_ایران_کشور_ملی_تیم,"[ایران, کشور, ملی, تیم, پیام, انتهای, انتهای پ...",[به گزارش ایسنا، حجت الاسلام موحدی آزاد دادستا...,ایران - کشور - ملی - تیم - پیام - انتهای - انت...,0.367262,False


In [33]:
topic_info[["Topic", "Count"]]

,Topic,Count
0,-1,1537
1,0,1015
2,1,467
3,2,270
4,3,252
5,4,187
6,5,180
7,6,145
8,7,124
9,8,114


In [39]:
n_docs = len(documents)
n_topics = len(topic_info) - 1
n_outliers = np.sum(np.array(topics) == -1)

print(f"Number of documents : {n_docs}")
print(f"Number of topics    : {n_topics}")
print(f"Outlier documents   : {n_outliers}")
print(f"Outlier percentage  : {100*n_outliers/n_docs:.2f}%")

Number of documents : 5969
Number of topics    : 44
Outlier documents   : 1537
Outlier percentage  : 25.75%


In [47]:
for topic in topic_info.Topic:

    if topic == -1:
        continue

    print("=" * 80)

    print(f"Topic {topic}")

    print(topic_model.get_topic(topic))

Topic 0
[('فیلم', 0.019580062171180284), ('کتاب', 0.01836706711512311), ('آثار', 0.014752007493731388), ('موسیقی', 0.013723242610700964), ('اثر', 0.013059025955017813), ('جشنواره', 0.013031008248922242), ('نمایش', 0.012866657914493715), ('میشود', 0.012297825582112978), ('تئاتر', 0.011402505608185195), ('هنر', 0.011020965379903443)]
Topic 1
[('نفت', 0.02807786498015879), ('شرکت', 0.0224958048447965), ('افزایش', 0.019444187523877575), ('دلار', 0.018535520329732123), ('هوش', 0.017388838902630564), ('مصنوعی', 0.017247305180603873), ('هوش مصنوعی', 0.017190181401879775), ('بشکه', 0.017013331857854374), ('آمریکا', 0.01683421966781469), ('کاهش', 0.015270530117700876)]
Topic 2
[('دولت', 0.019821041573463597), ('مجلس', 0.017156985871070826), ('کشور', 0.0145989747553339), ('شورای', 0.012191971043450107), ('دارو', 0.010683552471191203), ('بودجه', 0.010456176319961591), ('اقتصادی', 0.01038791931871779), ('ارز', 0.010061983592930835), ('افزایش', 0.009924040782363948), ('ادامه', 0.009799195031388977)

In [48]:
fig = topic_model.visualize_topics()

fig.show()

In [49]:
fig.write_html(
    output_dir / "topic_map.html"
)

In [50]:
fig = topic_model.visualize_barchart(
    top_n_topics=20
)

fig.show()

In [51]:
fig.write_html(
    output_dir / "topic_barchart.html"
)

In [52]:
fig = topic_model.visualize_hierarchy()

fig.show()

In [53]:
fig.write_html(
    output_dir / "topic_hierarchy.html"
)

In [54]:
fig = topic_model.visualize_heatmap()

fig.show()

In [55]:
fig.write_html(
    output_dir / "topic_heatmap.html"
)

In [56]:
hierarchical_topics = topic_model.hierarchical_topics(
    documents
)

hierarchical_topics.head()

100%|██████████| 43/43 [00:00<00:00, 189.90it/s]


,Parent_ID,Parent_Name,Topics,Child_Left_ID,Child_Left_Name,Child_Right_ID,Child_Right_Name,Distance
42,86,ایران_سال_پیام_کشور_میشود,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",83,تیم_بازی_کیلوگرم_وزن_تیم ملی,85,سال_کشور_میشود_درصد_ایران,2.231506
41,85,سال_کشور_میشود_درصد_ایران,"[0, 1, 2, 3, 4, 5, 7, 8, 10, 13, 16, 18, 19, 2...",82,درصد_افزایش_کاهش_کشور_دلار,84,ایران_فیلم_کتاب_اسلامی_آثار,1.599454
40,84,ایران_فیلم_کتاب_اسلامی_آثار,"[0, 4, 8, 10, 22, 23, 24, 26, 30, 33, 35, 38, 41]",79,ایران_فیلم_کتاب_اسلامی_آثار,59,خبرنگار_عماد_نعمتی_حنانه_کرمانی,1.458414
39,83,تیم_بازی_کیلوگرم_وزن_تیم ملی,"[6, 9, 11, 12, 14, 15, 17, 27, 31, 32, 36, 37,...",78,تیم_بازی_تیم ملی_بازیکنان_ایران,60,کیلوگرم_وزن_مدال_کشتی_دور,1.432321
38,82,درصد_افزایش_کاهش_کشور_دلار,"[1, 2, 3, 5, 7, 13, 16, 18, 19, 20, 21, 25, 28...",66,درصد_دلار_افزایش_کاهش_قیمت,81,کشور_آموزش_سازمان_استان_اشاره,1.409833


In [57]:
hierarchical_topics.to_csv(
    output_dir / "hierarchical_topics.csv",
    index=False,
    encoding="utf-8-sig"
)

In [58]:
fig = topic_model.visualize_hierarchy(
    hierarchical_topics=hierarchical_topics
)

fig.show()

In [59]:
fig.write_html(
    output_dir / "topic_tree.html"
)

In [60]:
topic_labels = {}

for topic in topic_info.Topic:

    if topic == -1:
        continue

    words = topic_model.get_topic(topic)

    label = " | ".join(
        [word for word, _ in words[:3]]
    )

    topic_labels[topic] = label

In [61]:
topic_labels

{0: 'فیلم | کتاب | آثار',
 1: 'نفت | شرکت | افزایش',
 2: 'دولت | مجلس | کشور',
 3: 'قضایی | قوه | قضاییه',
 4: 'اسلامی | انقلاب | ملت',
 5: 'برق | مصرف | آب',
 6: 'بازی | تیم | بازیکنان',
 7: 'آموزش | آموزشی | پرورش',
 8: 'سپاه | ارتش | عملیات',
 9: 'وزن | مدال | جهان',
 10: 'امام | شهید | انقلاب',
 11: 'ست | ۲۵ | ساعت',
 12: 'باشگاه | استقلال | پرسپولیس',
 13: 'میتواند | مصرف | بدن',
 14: 'تیم | بازی | تیم ملی',
 15: 'کشتی | المپیک | مدال',
 16: 'ماده | قانون | تسهیلات',
 17: 'تیم | تیم ملی | ایران',
 18: 'قیمتی | تغییرات | درصد',
 19: 'درصد | تورم | نرخ',
 20: 'آموزش | آموزش پرورش | پرورش',
 21: 'پزشکی | بهداشت | سلامت',
 22: 'کاربری شبکه | ایکس | حساب کاربری',
 23: 'خبرنگار | عماد | نعمتی',
 24: 'قهرمانی | کشتی | ایران',
 25: 'ثبتنام | آزمون | تحصیلی',
 26: 'میناب | ایران | خون',
 27: 'نتیجه | وزن | دور',
 28: 'شیوع | واکسن | جهانی بهداشت',
 29: 'توسعه | پست | کشور',
 30: 'دانش آموز | آموز | حادثه',
 31: 'امتیاز | ایران | گل',
 32: 'تیم | بازی | قهرمانی',
 33: 'ایران | آمریکا | جنگ'